# 01 Data Exploration

Inspect the base research panel, spot missing or sparse fields, and export a quick quality snapshot before factor construction.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "apps").exists():
            return candidate
    raise RuntimeError("repo root not found")


REPO_ROOT = locate_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from apps.quant_platform.research.data_loader import ResearchDataLoader

RESEARCH_ROOT = REPO_ROOT / "apps/quant_platform/research"
OUTPUT_ROOT = RESEARCH_ROOT / "output/notebook_exports"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
loader = ResearchDataLoader()
OUTPUT_ROOT

In [ ]:
panel = loader.load_panel(
    start_date="2024-01-01",
    end_date="2024-03-31",
    columns=[
        "ts_code", "trade_date", "open", "close", "close_qfq", "pct_chg",
        "turnover_rate_f", "volume_ratio", "pe_ttm", "pb", "total_mv", "circ_mv",
    ],
)
money_flow = loader.load_named_source(
    "money_flow",
    start_date="2024-01-01",
    end_date="2024-03-31",
    columns=["ts_code", "trade_date", "net_mf_amount", "buy_elg_amount", "sell_elg_amount"],
)
panel.head(), money_flow.head()

In [ ]:
quality_summary = pd.DataFrame(
    {
        "row_count": panel.count(),
        "missing_ratio": panel.isna().mean(),
        "unique_values": panel.nunique(dropna=True),
    }
).sort_values(["missing_ratio", "row_count"], ascending=[False, False])
quality_summary.head(20)

In [ ]:
daily_snapshot = (
    panel.groupby("trade_date", as_index=False)
    .agg(
        stock_count=("ts_code", "nunique"),
        avg_turnover=("turnover_rate_f", "mean"),
        avg_volume_ratio=("volume_ratio", "mean"),
        avg_pct_chg=("pct_chg", "mean"),
    )
)
quality_summary.to_csv(OUTPUT_ROOT / "01_quality_summary.csv")
daily_snapshot.to_csv(OUTPUT_ROOT / "01_daily_snapshot.csv", index=False)
daily_snapshot.tail()

## Next Checks

- If sparse fields dominate the top of `quality_summary`, confirm whether they are event-driven tables that should be shifted and expanded in the factor pipeline.
- If daily stock count changes sharply, inspect ST, suspension, or universe filters before moving on to factor analysis.
- Exported CSVs land in `research/output/notebook_exports/` for quick review.